# Skripsi pipeline — Google Colab runner

**One-time setup**
1. Upload the whole `CODE` folder to Google Drive at `MyDrive/Skripsi/CODE`
   (drag the folder into Drive in your browser; `.venv` is not needed — exclude it).
2. In Colab: `Runtime > Change runtime type > T4 GPU`.

**Daily workflow**: open this notebook, `Runtime > Run all`, leave the tab open.
When Colab times out, nothing is lost — every stage checkpoints to Drive
(KD generation per batch, training per epoch). The next `Run all` resumes
automatically from where it stopped.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/Skripsi/CODE'
import os
assert os.path.isdir(PROJECT), f'Upload the CODE folder to {PROJECT} first'
%cd {PROJECT}

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable it in Runtime settings')


In [ ]:
# Dependencies (torch, numpy, tqdm are preinstalled on Colab)
%pip install -q sentencepiece sacrebleu datasets transformers pyyaml 'numpy<2'


## Stage 1-3: data download, preprocessing, tokenizer
Each cell skips itself if its output already exists on Drive.


In [ ]:
import os
if not os.path.exists('data/raw/ccmatrix.tsv'):
    !python -m preprocessing_dataset.download_data
else:
    print('data/raw/ccmatrix.tsv exists — skipping download')


In [ ]:
if not os.path.exists('data/processed/train.tsv'):
    !python -m preprocessing_dataset.run_preprocessing
else:
    print('data/processed/train.tsv exists — skipping preprocessing')


In [ ]:
if not os.path.exists('data/processed/spm_en_id.model'):
    !python -m preprocessing_dataset.train_tokenizer
else:
    print('tokenizer exists — skipping')


## Stage 4: sequence-level KD data (the long one, ~1 day of GPU total)
Resumes from `kd_train.progress.json` automatically; rerun daily until it prints
`Already complete`.


In [ ]:
!python -m model.training.generate_kd_dataset


## Stage 5: hyperparameter search (once per architecture)
Comment back in the ones you still need; each saves `best_<arch>.json`.


In [ ]:
# !python -m model.training.hyperparameter_search --arch gru
# !python -m model.training.hyperparameter_search --arch lstm
# !python -m model.training.hyperparameter_search --arch transformer


## Stage 6: training
Set the experiment below. `--resume` continues the latest matching run,
or starts fresh if there is none — safe to rerun daily.


In [ ]:
ARCH = 'transformer'   # gru | lstm | transformer
DATA_MODE = 'kd'       # baseline | kd
QAT = False

qat_flag = '--qat' if QAT else ''
!python -m model.training.train --arch {ARCH} --data-mode {DATA_MODE} {qat_flag} --resume


## Stage 7: evaluate a finished run
Fill in the run folder name from `results/runs/`.


In [ ]:
RUN = ''  # e.g. 20260719-101500_transformer_kd
if RUN:
    !python -m model.evaluation.evaluate --checkpoint results/runs/{RUN}/best.pt
else:
    !ls -1 results/runs/ 2>/dev/null || echo 'no runs yet'


Quantization folding and CoreML conversion are **not** run here — CoreML
conversion needs macOS. Do stages 8-10 on your Mac; everything under
`results/` syncs back through Drive.
